In [39]:
import os
import pickle
import sys
from os.path import join

import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from Bio import SeqIO
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import roc_auc_score

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/Screening_and_application


In [40]:
df_test = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_test_with_ESM1b_ts.pkl")
)
df_test = df_test.loc[df_test["ESM1b"] != ""]
df_test.reset_index(inplace=True, drop=True)

df_train = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_train_with_ESM1b_ts.pkl")
)
df_train = df_train.loc[df_train["ESM1b"] != ""]
df_train.reset_index(inplace=True, drop=True)

/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)
/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)


In [ ]:
p450plant0 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant0.pkl")
)
p450plant1 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant1.pkl")
)
p450plant2 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant2.pkl")
)
p450plant3 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant3.pkl")
)
p450plant4 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant4.pkl")
)

p450plant = pd.concat(
    [p450plant0, p450plant1, p450plant2, p450plant3, p450plant4], ignore_index=True
)

In [42]:
p450plant

,enzyme,substrate,Binding,ECFP,ESM1b
0,CYP71AY5,GEI,1,0000000000000000000000000000000001000000000010...,"[-0.032707780599594116, 0.15956459939479828, -..."
1,CYP85A2,CAT,1,0100000010000000000000000000010001001000000000...,"[-0.12464731186628342, 0.25692933797836304, -0..."
2,CYP716A94,BAM,1,0000000000000000000000000000000101001000000000...,"[-0.1793026179075241, 0.2773324251174927, 0.02..."
3,CYP80B2,NME,1,0001000000000000000000000100000001000000100000...,"[-0.05789301171898842, 0.1279979795217514, -0...."
4,CYP85A69,DOC,1,0100000010000000000000000000000001001000000000...,"[-0.077804334461689, 0.2830597162246704, -0.00..."
...,...,...,...,...,...
1546,CYP71D495,LUP,0,0000000000000000000000000000000101001000000000...,"[-0.08674226701259613, 0.1240856796503067, -0...."
1547,CYP716A155,Vincadifformine,0,0000100000000000000000000000000001001000000000...,"[-0.16968412697315216, 0.2657066881656647, 0.0..."
1548,CYP716A155,KEO,0,0000100000000000000000000000000001011000000000...,"[-0.16968412697315216, 0.2657066881656647, 0.0..."
1549,CYP706C55,LIT,0,0100000000000000000000000010000001000000011000...,"[-0.07378055155277252, 0.1300899088382721, 0.0..."


In [43]:
# deletedataname = "Arabidopsis thaliana"
# deletedataname = "Erigeron_breviscapus"
deletedataname = "Glycine_max"
# deletedataname = "Zea_mays"

In [ ]:
deletedata = pd.read_csv(
    our_data + "screening1/P450_Substrate.txt",
    sep="\t",
    header=None,
    names=["enzyme", "substrate", "sequence"],
)
deletedata = deletedata.sample(frac=1, random_state=42)

! python distinctonly.py --input "/home/hanxd/Repositories/ESP/data/our_data/screening2/{deletedataname}.fasta" --output "/home/hanxd/Repositories/ESP/data/our_data/screening2/{deletedataname}_q.fasta"

In [45]:
fasta_file = our_data + f"screening2/{deletedataname}_q.fasta"
sequences = SeqIO.to_dict(SeqIO.parse(fasta_file, "fasta"))

In [46]:
for index, row in deletedata.iterrows():
    nowenzyme = row["enzyme"]
    nowsubstrate = row["substrate"]
    try:
        target_sequence = sequences[nowenzyme].seq
    except:
        target_sequence = ""
    deletedata.loc[index, "sequence"] = str(target_sequence)

In [ ]:
deletedata = deletedata[deletedata["sequence"] != ""]
has_null_values = deletedata.isna().any().any()
has_empty_strings = deletedata.applymap(lambda x: x == "")
if has_empty_strings.any().any():
    print("DataFrame contains empty strings.")
if has_null_values:
    print("DataFrame contains empty values.")

In [48]:
deletedata["ESM1b"] = ""
deletedata["ECFP"] = ""

rep_dict = join(CURRENT_DIR, "..", "data", "our_data", "screening1/esm/")

for ind in deletedata.index:
    esms = torch.load(rep_dict + str(deletedata["enzyme"][ind]) + ".pt")
    sdf_file_path = (
        our_data + "screening1/SDF_300/" + deletedata["substrate"][ind] + ".sdf"
    )
    mol = Chem.MolFromMolFile(sdf_file_path)
    ecfpso = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024).ToBitString()
    deletedata["ESM1b"][ind] = esms["mean_representations"][33].tolist()
    deletedata["ECFP"][ind] = ecfpso

In [ ]:
deletedata["Binding"] = 1
has_null_values = deletedata.isna().any().any()
has_empty_strings = deletedata.applymap(lambda x: x == "")
if has_empty_strings.any().any():
    print("DataFrame contains empty strings.")
if has_null_values:
    print("DataFrame contains empty values.")

In [50]:
deletedata.to_pickle(our_data + "screening2/" + f"{deletedataname}_deletedata.pkl")

In [51]:
deletedata

,enzyme,substrate,sequence,ESM1b,ECFP,Binding
403,CYP72A69,SOY,MEAAWVNILMLILILALIWVWKKFNSLWLTPKRLEKILREQGLRGS...,"[-0.11592403054237366, 0.1274290382862091, -0....",0000000000000000000000000000000001001000000000...,1
182,CYP93B16,NAR,MISESLLLVFLIVFISASLLKLLFVRENKPKAHLKNPPSPPAIPII...,"[-0.04198030009865761, 0.13522382080554962, -0...",0100000000000000000000000000000000000000000000...,1
245,CYP71A10,fluometuron,MALLSSVLKQLPHELSSTHYLTVFFCIFLILLQLIRRNKYNLPPSP...,"[-0.07291638851165771, 0.12347349524497986, -0...",0000000000000001000000000000000001000000000000...,1
280,CYP71D9,LIQ,MDLQLLYFTSIFSIFIFMFMTHKIVTKKSNSTPSLPPGPWKLPIIG...,"[-0.0921514704823494, 0.1621680110692978, -0.0...",0100000000000000000000000000000000000000000000...,1
183,CYP93A1,DIT,MAYQVLLICLVSTIVFAYILWRKQSKKNLPPSPKALPIIGHLHLVS...,"[-0.036078907549381256, 0.11775028705596924, -...",0001000000000001000000000000000000000000000000...,1
178,CYP93C1,LIQ,MLLELALGLFVLALFLHLRPTPSAKSKALRHLPNPPSPKPRLPFIG...,"[-0.04426438733935356, 0.16090595722198486, 0....",0100000000000000000000000000000000000000000000...,1
190,CYP93E1,BAM,MLDIKGYLVLFFLWFISTILIRSIFKKPQRLRLPPGPPISVPLLGH...,"[-0.033789537847042084, 0.13793006539344788, -...",0000000000000000000000000000000101001000000000...,1
91,CYP81E100,DAI,MEILSFLSYSLLYVFLFLLLKLLLQARKFQNLPPGPPSLPIIGNLH...,"[-0.014800388365983963, 0.07554430514574051, -...",0000000000000000000000000000000000000000000000...,1
52,CYP82D26,NAR,MIIHYQNQSYPTTSSDTMLHMENPLLQSTNSITFTFFGLLLFLFVL...,"[-0.05101926997303963, 0.07534659653902054, -0...",0100000000000000000000000000000000000000000000...,1


In [52]:
len(deletedata)

9

In [ ]:
filtered_rows = p450plant[
    (p450plant["Binding"] == 1) & p450plant["enzyme"].isin(deletedata["enzyme"])
]

p450plant_filtered = p450plant.drop(filtered_rows.index)
p450plant_filtered.reset_index(drop=True, inplace=True)

In [54]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


train_X, train_y = create_input_and_output_data(df=df_train)
test_X, test_y = create_input_and_output_data(df=df_test)
train_new_X, train_new_y = create_input_and_output_data(df=p450plant_filtered)

train_X = np.concatenate([train_X, train_new_X])
train_y = np.concatenate([train_y, train_new_y])

feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

In [55]:
param = {
    "learning_rate": 0.31553117247348733,
    "max_delta_step": 1.7726044219753656,
    "max_depth": 10,
    "min_child_weight": 1.3845040588450772,
    "num_rounds": 342.68325188584106,
    "reg_alpha": 0.531395259755843,
    "reg_lambda": 3.744980563764689,
    "weight": 0.26187490421514203,
}


num_round = param["num_rounds"]
param["objective"] = "binary:logistic"

weights = np.array(
    [param["weight"] if binding == 0 else 1.0 for binding in np.array(train_y)]
)

del param["num_rounds"]
del param["weight"]

dtrain = xgb.DMatrix(
    np.array(train_X),
    weight=weights,
    label=np.array(train_y),
    feature_names=feature_names,
)
dtest = xgb.DMatrix(
    np.array(test_X), label=np.array(test_y), feature_names=feature_names
)
dtrain_p450 = xgb.DMatrix(
    np.array(train_new_X), label=np.array(train_new_y), feature_names=feature_names
)

evallist = [(dtrain, "train"), (dtrain_p450, "dtrain_p450")]
bst = xgb.train(param, dtrain, int(num_round), evallist, verbose_eval=10)
y_test_pred = np.round(bst.predict(dtest))
acc_test = np.mean(y_test_pred == np.array(test_y))
roc_auc = roc_auc_score(np.array(test_y), bst.predict(dtest))

print("Accuracy on test set: %s, ROC-AUC score for test set: %s" % (acc_test, roc_auc))

[0]	train-error:0.26941	dtrain_p450-error:0.54993
[10]	train-error:0.13848	dtrain_p450-error:0.38975
[20]	train-error:0.09250	dtrain_p450-error:0.25162
[30]	train-error:0.06239	dtrain_p450-error:0.15305
[40]	train-error:0.04560	dtrain_p450-error:0.12451
[50]	train-error:0.03490	dtrain_p450-error:0.09728
[60]	train-error:0.02845	dtrain_p450-error:0.08496
[70]	train-error:0.02345	dtrain_p450-error:0.07134
[80]	train-error:0.01887	dtrain_p450-error:0.05772
[90]	train-error:0.01571	dtrain_p450-error:0.04475
[100]	train-error:0.01329	dtrain_p450-error:0.03956
[110]	train-error:0.01143	dtrain_p450-error:0.03567
[120]	train-error:0.00973	dtrain_p450-error:0.02918
[130]	train-error:0.00834	dtrain_p450-error:0.02075
[140]	train-error:0.00734	dtrain_p450-error:0.02010
[150]	train-error:0.00649	dtrain_p450-error:0.01881
[160]	train-error:0.00563	dtrain_p450-error:0.01881
[170]	train-error:0.00485	dtrain_p450-error:0.01492
[180]	train-error:0.00437	dtrain_p450-error:0.01427
[190]	train-error:0.003

In [56]:
pickle.dump(
    bst, open(join(our_data + f"screening2/{deletedataname}deletedatamodel.dat"), "wb")
)

In [ ]:
len(p450plant) - len(train_new_y)

9